<a href="https://colab.research.google.com/github/addulamalerupareddy-gif/Data-Analatics/blob/main/Relationships_Function_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, chi2_contingency, f_oneway

# ---------- Helper: Strength Category ----------
def get_strength(value):
    v = abs(value)
    if v < 0.3:
        return "Weak"
    elif v < 0.7:
        return "Moderate"
    else:
        return "Strong"


# ---------- Main Function ----------
def relationship_table(df, cat_threshold=20):

    results = []

    # Detect column types
    numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
    categorical_cols = [col for col in df.columns
                        if col not in numeric_cols or df[col].nunique() <= cat_threshold]

    # 1️⃣ NUMERIC vs NUMERIC → Spearman
    for i, colA in enumerate(numeric_cols):
        for colB in numeric_cols[i+1:]:
            val, p = spearmanr(df[colA], df[colB])
            results.append({
                "Column A": colA,
                "Column B": colB,
                "Test": "Spearman Correlation",
                "Statistic": round(val, 4),
                "p-value": round(p, 4),
                "Strength": get_strength(val)
            })

    # 2️⃣ CATEGORICAL vs CATEGORICAL → Chi-Square
    for i, colA in enumerate(categorical_cols):
        for colB in categorical_cols[i+1:]:
            table = pd.crosstab(df[colA], df[colB])
            chi2, p, dof, expected = chi2_contingency(table)
            results.append({
                "Column A": colA,
                "Column B": colB,
                "Test": "Chi-Square Test",
                "Statistic": round(chi2, 4),
                "p-value": round(p, 4),
                "Strength": "Strong" if p < 0.05 else "Weak"
            })

    # 3️⃣ CATEGORICAL vs NUMERIC → ANOVA
    for colA in categorical_cols:
        for colB in numeric_cols:
            groups = [df[colB][df[colA] == cat] for cat in df[colA].dropna().unique()]

            if len(groups) > 1:
                f, p = f_oneway(*groups)
                results.append({
                    "Column A": colA,
                    "Column B": colB,
                    "Test": "ANOVA",
                    "Statistic": round(f, 4),
                    "p-value": round(p, 4),
                    "Strength": "Strong" if p < 0.05 else "Weak"
                })

    return pd.DataFrame(results)


In [ ]:
df = pd.read_csv('/content/customer_churn_dataset-testing-master.csv')
rel_table = relationship_table(df)
rel_table

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)


,Column A,Column B,Test,Statistic,p-value,Strength
0,CustomerID,Age,Spearman Correlation,0.0435,0.0000,Weak
1,CustomerID,Tenure,Spearman Correlation,0.1021,0.0000,Weak
2,CustomerID,Usage Frequency,Spearman Correlation,-0.0639,0.0000,Weak
3,CustomerID,Support Calls,Spearman Correlation,0.1615,0.0000,Weak
4,CustomerID,Payment Delay,Spearman Correlation,0.2889,0.0000,Weak
...,...,...,...,...,...,...
86,Churn,Support Calls,ANOVA,6584.8002,0.0000,Strong
87,Churn,Payment Delay,ANOVA,29012.6272,0.0000,Strong
88,Churn,Total Spend,ANOVA,402.9040,0.0000,Strong
89,Churn,Last Interaction,ANOVA,0.5112,0.4746,Weak
